# Track 1 · 07 EEGNet — the organisers' own recipe, on the harness's own data

**One job.** Show a training run that learns: train loss falling *and* validation
improving, using a recipe we did not invent.

Self-contained: upload this one file. The EEG and target table come from Drive.

---

## Why this, after fourteen REVE configurations

Two things surfaced from reading the competition repo rather than the neuralbench
model configs.

**The harness never uses REVE's preprocessing.** `benchmark_utils/nb_task.py` builds
the scoring data with `merge_task_config(modality, task, dataset)` — no model is passed,
so `reve.yaml`'s overrides never apply at evaluation. Every submission receives the
*default* pipeline. We built everything on REVE's 200 Hz pipeline instead, so any REVE
submission would have been scored on data prepared differently from its training data.

**The organisers ship a baseline we never ran.** `tracks/image_decoding/solvers/
eegnet_clip.py`: EEGNet trained end to end on raw EEG with a standard CLIP loss. That
matches what our diagnostics kept saying — raw samples beat frozen REVE features, and
the signal needs channel-specific weighting, which is exactly what EEGNet's depthwise
spatial filters learn.

## The recipe, from source

Preprocessing is the harness's, read from `neuralbench/defaults/config.yaml`, the image
task config and `neuralset/extractors/neuro.py`, in that code's order:

| step | value |
|---|---|
| notch | 50 and 60 Hz plus harmonics below Nyquist: 50, 60, 100, 120 |
| band-pass | 0.1 – 75 Hz, at the native 256 Hz |
| resample | 120 Hz |
| scale | sklearn `RobustScaler`, per channel over the whole recording |
| window | −0.2 → 0.8 s, 120 samples |
| baseline | subtract the mean of the pre-stimulus 0.2 s |
| clamp | ±20 |

Model and loss are `eegnet_clip.py` verbatim: `braindecode.models.EEGNet`, Adam at
1e-3, batch 64, and

```python
logits = normalize(pred) @ normalize(target).T / 0.07
```

That loss is also the one cell of our grid never tested. Our runs used neuralbench's
`ClipLoss` with `temperature: false`, which neuraltrain implements as a frozen
`exp(0.0)` — tau = 1 — with only the target normalised. Normalising both sides keeps
predictions bounded, and tau = 0.07 keeps the loss range wide.

One deliberate addition: windows showing the same image are masked as negatives of each
other, since otherwise the loss punishes the model for being right.

## Published reference

EEGNet reaches **28.13% top-5** on THINGS-EEG2 in NeuralBench. Alljoined-1.6M is
consumer-grade Emotiv hardware and should be harder, but the hidden sealed cohort uses
the same hardware, so this is the right corpus to be measured on.

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

# Preprocessing: the harness's default pipeline. Do not tune these; matching them is
# the point, since a model trained on anything else is scored on data it never saw.
SFREQ = 120.0
NOTCH = (50.0, 60.0)
BANDPASS = (0.1, 75.0)
TMIN, DURATION = -0.2, 1.0
BASELINE = (-0.2, 0.0)          # relative to stimulus onset
CLAMP = 20.0

# Model and loss: eegnet_clip.py, with regularisation turned up after the first run
# learned (test top-5 4.09%, rank 0.5673) but plateaued by epoch ~18 while train loss
# kept falling. Width is left at the default on purpose: widening adds capacity, and
# the model is already mildly overfitting.
EPOCHS = 80          # was 30. The bigger batch gives 4x fewer updates per epoch
                     # (42 vs 168), so more epochs keep the total step count
                     # comparable; the best checkpoint is kept either way.
BATCH = 256          # was 64. More in-batch negatives, closer to the 200-image test
LR = 1e-3            # unchanged
DROPOUT = 0.5        # was 0.25 (EEGNet default)
WEIGHT_DECAY = 1e-4  # was none. Very small, via AdamW
TEMPERATURE = 0.07

PREVIOUS = dict(rank=0.5673, top5=4.09, top5_agg=6.04)   # first run, for comparison

RUN_CONTROL = True
MAKE_ZIP = True                 # package the trained model at the end              # positive control first, so a flat result is readable
VAL_IMAGE_FRAC = 0.2
SEED = 33

In [ ]:
%pip install -q mne 'braindecode>=1.8'

import importlib
for mod in ('mne', 'braindecode', 'torch', 'sklearn'):
    print(f'{mod:12} {getattr(importlib.import_module(mod), "__version__", "ok")}')

In [ ]:
import json, time, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, pandas as pd, mne, torch, torch.nn.functional as F
mne.set_log_level('ERROR')

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

ROOT = Path(ROOT)
BASE = ROOT / 'Track1'
PREP, RAW, WEIGHTS = BASE / 'prepared', BASE / 'raw', BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
DATASET = 'nm000134'
STIM = {'train': 'stim_train', 'test': 'stim_test'}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def l2(a):
    return a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-12)


def rank_pct(pred, y_ids, emb):
    """Mean fraction of the gallery ranked below the true image. 0.5 is chance."""
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2(pred) @ l2(emb[gal]).T
    tc = sims[np.arange(len(sims)), truth]
    return float((sims < tc[:, None]).mean())


def top5(pred, y_ids, emb):
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2(pred) @ l2(emb[gal]).T
    k = min(5, len(gal))
    tk = np.argpartition(-sims, kth=k - 1, axis=1)[:, :k]
    return 100.0 * np.mean([t in r for t, r in zip(truth, tk)]), 100.0 * k / len(gal)


def rank_null(pred, y_ids, n=200):
    """Validation rank under permuted labels, for these exact predictions.

    Validation loss can drift down with no signal at all (the fixture with random
    targets did), and "the best epoch was not the first" happens by chance, so
    neither is evidence. This is.
    """
    gal = np.unique(y_ids)
    pos = {g: i for i, g in enumerate(gal)}
    truth = np.array([pos[y] for y in y_ids])
    sims = l2(pred) @ l2(emb[gal]).T
    rows = np.arange(len(sims))
    return np.array([float((sims < sims[rows, p][:, None]).mean())
                     for p in (rng.permutation(truth) for _ in range(n))])


print(f'device {DEVICE}')

## Step 1 — preprocess exactly as the harness does

Filtering and scaling run on the continuous recording, then windows are cut, baselined
and clamped. The order follows `neuralset/extractors/neuro.py`.

In [ ]:
t = np.load(PREP / 'image_embeddings.npz', allow_pickle=True)
emb = l2(t['emb'].astype('float32'))
DIM = emb.shape[1]
key = {k: i for i, k in enumerate(t['keys'].tolist())}
man = json.loads((PREP / 'prepare_manifest.json').read_text())

N_TIMES = int(round(DURATION * SFREQ))
PRE = int(round(-TMIN * SFREQ))
B0, B1 = (int(round((b - TMIN) * SFREQ)) for b in BASELINE)


def harness_windows(rel, split):
    raw = mne.io.read_raw_edf(RAW / DATASET / rel, preload=True)
    raw.pick('eeg')
    nyq = raw.info['sfreq'] / 2
    freqs = sorted({f for base in NOTCH
                    for f in np.arange(base, min(nyq, 301), base)})
    raw.notch_filter(freqs, phase='zero', filter_length='auto', verbose=False)
    lo, hi = BANDPASS
    raw.filter(lo, hi if hi < nyq else None, verbose=False)   # neuralset drops a lowpass at or above Nyquist
    raw.resample(SFREQ, verbose=False)
    d = raw.get_data()
    # sklearn RobustScaler per channel over the whole recording: (x - median) / IQR
    med = np.median(d, axis=1, keepdims=True)
    iqr = np.subtract(*np.percentile(d, [75, 25], axis=1))[:, None]
    iqr[iqr == 0] = 1.0
    d = ((d - med) / iqr).astype('float32')

    df = pd.read_csv(RAW / DATASET / rel.replace('_eeg.edf', '_events.tsv'), sep='\t')
    df = df[df.trial_type == STIM[split]].dropna(subset=['stim_file']).reset_index(drop=True)
    s = np.round(df.onset.values * SFREQ).astype(int) - PRE     # window start samples
    ok = (s >= 0) & (s + N_TIMES <= d.shape[1])
    df, s = df[ok].reset_index(drop=True), s[ok]
    X = np.stack([d[:, i:i + N_TIMES] for i in s])
    X = X - X[:, :, B0:B1].mean(axis=2, keepdims=True)          # pre-stimulus baseline
    return np.clip(X, -CLAMP, CLAMP), df.stim_file.values


t0 = time.time()
parts = {'train': [[], [], []], 'test': [[], [], []]}
for rel, meta in sorted(man['recordings'].items()):
    X, stim = harness_windows(rel, meta['split'])
    idx = np.array([key.get(s_, -1) for s_ in stim])
    k = idx >= 0
    for j, arr in enumerate((X[k], idx[k], np.full(int(k.sum()), meta['subject']))):
        parts[meta['split']][j].append(arr)
CH = mne.io.read_raw_edf(RAW / DATASET / sorted(man['recordings'])[0],
                         preload=False).pick('eeg').ch_names
Wtr = [np.concatenate(a) for a in parts['train']]
Wte = [np.concatenate(a) for a in parts['test']]
N_CHANS = Wtr[0].shape[1]

val_imgs = set(rng.choice(np.unique(Wtr[1]), int(len(np.unique(Wtr[1])) * VAL_IMAGE_FRAC),
                          replace=False).tolist())
IV = np.array([y in val_imgs for y in Wtr[1]])
print(f'train {Wtr[0].shape}   test {Wte[0].shape}   ({time.time() - t0:.0f}s)')
print(f'{(~IV).sum():,} fit / {IV.sum():,} val windows, {len(np.unique(Wte[1]))} test images')
print(f'value range after clamp: {Wtr[0].min():.1f} .. {Wtr[0].max():.1f}, '
      f'sd {Wtr[0].std():.2f}')

In [ ]:
from braindecode.models import EEGNet

EMB_T = torch.tensor(emb, device=DEVICE)


def clip_loss(pred, idx):
    """eegnet_clip.py's loss, plus masking of same-image pairs within a batch."""
    logits = F.normalize(pred, dim=-1) @ F.normalize(EMB_T[idx], dim=-1).T / TEMPERATURE
    same = idx[:, None] == idx[None, :]
    same.fill_diagonal_(False)
    logits = logits.masked_fill(same, -1e4)
    return F.cross_entropy(logits, torch.arange(len(pred), device=pred.device))


def train(Xtr, ytr, Xv, yv, epochs, label, quiet=False):
    net = EEGNet(n_chans=N_CHANS, n_outputs=DIM, n_times=N_TIMES,
                 drop_prob=DROPOUT).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    Xtr_t = torch.tensor(Xtr); ytr_t = torch.tensor(ytr.astype('int64'))
    Xv_t = torch.tensor(Xv);   yv_t = torch.tensor(yv.astype('int64'))
    n_par = sum(p.numel() for p in net.parameters())

    @torch.no_grad()
    def infer(X):
        net.eval()
        return torch.cat([net(X[i:i + 256].to(DEVICE)) for i in range(0, len(X), 256)])

    if not quiet:
        print(f'{label}: EEGNet {n_par / 1e3:.0f}k parameters   '
              f'chance loss {np.log(BATCH):.4f}')
    best, best_state, hist = -1, None, []
    for ep in range(1, epochs + 1):
        net.train()
        perm = torch.randperm(len(Xtr_t))
        tot, t0 = 0.0, time.time()
        for i in range(0, len(perm) - 7, BATCH):
            b = perm[i:i + BATCH]
            opt.zero_grad()
            loss = clip_loss(net(Xtr_t[b].to(DEVICE)), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        pv = infer(Xv_t)
        vl = float(np.mean([float(clip_loss(pv[i:i + BATCH], yv_t[i:i + BATCH].to(DEVICE)))
                            for i in range(0, len(pv) - 7, BATCH)]))
        r = rank_pct(pv.cpu().numpy(), yv, emb)
        hist.append((ep, tot / len(Xtr_t), vl, r))
        if r > best:
            best = r
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(Xtr_t):.4f}  val loss {vl:.4f}  '
                  f'val rank {r:.4f}{"  *" if r == best else ""}  {time.time() - t0:.0f}s')
    net.load_state_dict(best_state)
    return net, hist, infer


print('ready')

## Step 2 — positive control

A target-derived pattern is planted into a copy of the windows at +150 ms, and the same
training loop runs on it. If this does not learn, the loop is broken and nothing in
Step 3 means anything. If it does, a flat Step 3 is a statement about the data.

In [ ]:
if RUN_CONTROL:
    proj = rng.standard_normal((DIM, N_CHANS)).astype('float32')

    def plant(X, y):
        Z = X.copy()
        a = PRE + int(round(0.15 * SFREQ)); b = a + int(round(0.10 * SFREQ))
        pat = emb[y] @ proj
        pat /= pat.std() + 1e-12
        Z[:, :, a:b] += 2.0 * Z.std() * pat[:, :, None] * np.hanning(b - a)[None, None, :]
        return Z.astype('float32')

    sub = rng.choice(np.where(~IV)[0], min(4000, (~IV).sum()), replace=False)
    _, hist_c, _ = train(plant(Wtr[0][sub], Wtr[1][sub]), Wtr[1][sub],
                         plant(Wtr[0][IV], Wtr[1][IV]), Wtr[1][IV], 30, 'control', quiet=True)
    first, last = hist_c[0][3], max(h[3] for h in hist_c)
    print(f'\ncontrol val rank {first:.4f} -> {last:.4f}')
    print('CONTROL PASSES: the loop learns a planted signal' if last > 0.6 else
          'CONTROL FAILS: the loop cannot learn a planted signal; fix before Step 3')
else:
    print('control skipped')

## Step 3 — the real run

What to look for is a **trend**, not a number: train loss falling while validation loss
holds or falls and validation rank climbs. Every run so far had train loss falling and
validation getting worse from epoch 1.

In [ ]:
net, hist, infer = train(Wtr[0][~IV], Wtr[1][~IV], Wtr[0][IV], Wtr[1][IV], EPOCHS, 'real')

best_ep = max(hist, key=lambda h: h[3])
vl_min = min(hist, key=lambda h: h[2])
print(f'\nbest val rank {best_ep[3]:.4f} at epoch {best_ep[0]}; '
      f'lowest val loss {vl_min[2]:.4f} at epoch {vl_min[0]}')

pred = infer(torch.tensor(Wte[0])).cpu().numpy()
r_te = rank_pct(pred, Wte[1], emb)
a_te, ch = top5(pred, Wte[1], emb)
agg = {}
for p_, y_, s_ in zip(pred, Wte[1], Wte[2]):
    agg.setdefault((s_, y_), []).append(p_)
ks = sorted(agg)
a_agg, _ = top5(np.stack([np.mean(agg[k_], 0) for k_ in ks]), np.array([k_[1] for k_ in ks]), emb)
print(f'\n  {"EEGNet-CLIP":<16} rank {r_te:.4f}   top-5 {a_te:5.2f}% (per-window, = warm-up metric)'
      f'   {a_agg:5.2f}% (subject-agg)   chance {ch:.2f}%')
print(f'  {"previous EEGNet":<16} rank {PREVIOUS["rank"]:.4f}   top-5  {PREVIOUS["top5"]:.2f}%'
      f'                                  {PREVIOUS["top5_agg"]:.2f}% (subject-agg)')
print(f'  {"raw EEG ridge":<16} rank 0.5116   top-5  2.76%')
print(f'  {"REVE unpooled":<16} rank 0.5070   top-5  2.67%')
print(f'  {"REVE pooled":<16} rank 0.4998   top-5  2.48%')

pv_best = infer(torch.tensor(Wtr[0][IV])).cpu().numpy()
nullv = rank_null(pv_best, Wtr[1][IV])
thr = float(np.percentile(nullv, 95))
climb = best_ep[3] - hist[0][3]
print(f'\nval rank under permuted labels: mean {nullv.mean():.4f}  sd {nullv.std():.4f}  '
      f'95th {thr:.4f}')
print(f'best val rank {best_ep[3]:.4f} ({(best_ep[3] - nullv.mean()) / max(nullv.std(), 1e-9):.1f} '
      f'sd above the null), climbed {climb:+.4f} from epoch 1')
print()
TRENDING = bool(best_ep[3] > thr and best_ep[0] > 1 and climb > 2 * nullv.std())
if TRENDING:
    print(f'Trending: validation rank climbed to epoch {best_ep[0]} and clears its own')
    print('null. The model is learning something that generalises to unseen images.')
elif best_ep[3] > thr:
    print('Above the null, but the best epoch is the first or the climb is inside the')
    print('noise, so there is signal without a clear learning trend.')
else:
    print('Not trending: validation rank never clears its own null. Whatever the loss')
    print('curves look like, the model has not learned anything about unseen images.')

torch.save(net.state_dict(), WEIGHTS / 'eegnet_clip_weights.pt')
(WEIGHTS / 'eegnet_clip_config.json').write_text(json.dumps(dict(
    n_chans=N_CHANS, n_times=N_TIMES, n_outputs=DIM, sfreq=SFREQ,
    test_rank=r_te, test_top5=a_te, best_epoch=best_ep[0]), indent=2))
print(f'\nsaved -> {WEIGHTS / "eegnet_clip_weights.pt"}')

## Step 4 — package for submission

The zip is `weights.pt`, the solver and a small config. The solver matches incoming
electrodes to the training electrodes **by name**: extra electrodes are ignored,
training electrodes the input lacks are filled with zeros, and it prints how many
matched into the Codabench log. That lets the same file run on the 63-channel warm-up
cap and on the 32-channel sealed cohort, and it also protects against the sealed data
listing the same electrodes in a different order.

Before writing the zip it is loaded back the way the harness loads it and checked at
four geometries, including that shuffling the input order gives *identical* output.


In [ ]:
# Source of truth is solvers/track1_image_eegnet_clip.py in the repo.
SUBMISSION_PY = "\"\"\"Track 01 submission: EEGNet trained with a CLIP retrieval loss.\n\nThe model and loss follow the organisers' `eegnet_clip.py` baseline; the weights\ncome from `track1_07_eegnet.ipynb`, trained on Alljoined-1.6M with the harness's\nown preprocessing, so the scoring data is prepared exactly as the training data\nwas.\n\nEEGNet learns one spatial weight per electrode, in the order it was trained on.\nThe scoring data may come from a different headset (the warm-up uses a 63-channel\ncap) or list the same electrodes in a different order. Inputs are therefore\nmatched to the training electrodes by name. Electrodes the model never saw are\nignored, and training electrodes absent from the input are filled with zeros:\nafter the harness's scaling and baseline a real channel sits near zero, so zero\nreads as \"no information\" rather than as signal.\n\"\"\"\n\nimport json\n\nimport torch\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import EEGNet\n\n\ndef resample_fft(x, n_out):\n    \"\"\"Resample along time via the rfft spectrum. Pure torch, since nothing is\n    installed at evaluation.\"\"\"\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass ChannelMatchedEEGNet:\n\n    def __init__(self, cfg, in_names, in_sfreq, device):\n        self.device = device\n        self.cfg = cfg\n        self.n_times = int(cfg[\"n_times\"])\n        self.sfreq = float(cfg[\"sfreq\"])\n        self.in_sfreq = float(in_sfreq)\n        self.net = EEGNet(\n            n_chans=len(cfg[\"channels\"]),\n            n_outputs=int(cfg[\"n_outputs\"]),\n            n_times=self.n_times,\n        ).to(device)\n\n        # For each training electrode, the index of the same electrode in the\n        # input, or -1 if the input does not have it. Case is folded because\n        # montages disagree on it (FP1 vs Fp1).\n        lookup = {name.casefold(): i for i, name in enumerate(in_names)}\n        self.src = [lookup.get(name.casefold(), -1) for name in cfg[\"channels\"]]\n        matched = [n for n, i in zip(cfg[\"channels\"], self.src) if i >= 0]\n        missing = [n for n, i in zip(cfg[\"channels\"], self.src) if i < 0]\n        print(\n            f\"[EEGNet-CLIP] matched {len(matched)}/{len(cfg['channels'])} training \"\n            f\"channels by name among {len(in_names)} inputs; \"\n            f\"zero-filled: {missing if missing else 'none'}\"\n        )\n        if not matched:\n            raise RuntimeError(\n                f\"no training channel found in the input: {list(in_names)[:8]}\"\n            )\n\n    @torch.no_grad()\n    def predict(self, X):\n        self.net.eval()\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(\n            X.shape[0], len(self.src), X.shape[-1], device=self.device\n        )\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        # Match the training rate and window length. At the harness's own\n        # settings both are no-ops.\n        if self.in_sfreq != self.sfreq:\n            out = resample_fft(\n                out, int(round(out.shape[-1] * self.sfreq / self.in_sfreq))\n            )\n        if out.shape[-1] > self.n_times:\n            out = out[..., : self.n_times]\n        elif out.shape[-1] < self.n_times:\n            out = F.pad(out, (0, self.n_times - out.shape[-1]))\n        return self.net(out)\n\n\nclass Solver(CompetSolver):\n\n    name = \"EEGNet-CLIP-Alljoined\"\n\n    requirements = [\"pip::braindecode\"]\n\n    def load_model(self, meta):\n        sub = meta[\"submission_dir\"]\n        cfg = json.loads((sub / \"eegnet_config.json\").read_text())\n        in_names = meta.get(\"ch_names\") or [c[\"ch_name\"] for c in meta[\"chs_info\"]]\n        model = ChannelMatchedEEGNet(cfg, in_names, meta[\"sfreq\"], meta[\"device\"])\n        state = torch.load(\n            sub / \"weights.pt\", map_location=meta[\"device\"], weights_only=True\n        )\n        model.net.load_state_dict(state)\n        return model\n"


In [ ]:
if MAKE_ZIP and not (TRENDING or globals().get('FORCE_ZIP')):
    print('Not packaging: the run did not clear its null. Set FORCE_ZIP = True to override.')
elif MAKE_ZIP:
    import importlib.util, shutil, sys, types
    out = BASE / 'submissions' / 'track1_eegnet_clip'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    torch.save({k: v.cpu() for k, v in net.state_dict().items()}, out / 'weights.pt')
    (out / 'eegnet_config.json').write_text(json.dumps(dict(
        channels=list(CH), n_times=N_TIMES, sfreq=SFREQ, n_outputs=DIM,
        trained_on='Alljoined-1.6M (nm000134), harness default preprocessing',
        test_rank=r_te, test_top5=a_te), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)

    # Load it the way the harness does, with a stand-in for the base class.
    sub = types.ModuleType('benchmark_utils.base_solver')
    sub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = sub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def run(names, X, sfreq=SFREQ):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None,
                                     'sfreq': sfreq, 'submission_dir': out,
                                     'device': DEVICE, 'n_outputs': DIM})
        return m.predict(torch.tensor(X)).cpu().numpy()

    Xs = Wte[0][:8]
    base = run(CH, Xs)
    perm = rng.permutation(len(CH))
    shuffled = run([CH[i] for i in perm], Xs[:, perm])
    extra = [f'X{i}' for i in range(31)]                 # stand-ins for a 63-channel cap
    wide_names = list(CH) + extra
    wide = np.concatenate([Xs, rng.standard_normal((8, 31, N_TIMES)).astype('float32')], 1)
    wide_out = run(wide_names, wide)
    drop = [n for n in CH if n not in ('Oz', 'Fp1')]      # two training channels absent
    part = run(drop, Xs[:, [CH.index(n) for n in drop]])

    checks = [
        ('trained geometry', base.shape == (8, DIM) and np.isfinite(base).all(), f'{base.shape}'),
        ('order-independent', np.allclose(base, shuffled, atol=1e-5),
         f'max |diff| {np.abs(base - shuffled).max():.1e} after shuffling electrodes'),
        ('63-channel input', np.allclose(base, wide_out, atol=1e-5),
         'extra electrodes ignored, output identical'),
        ('missing electrodes', part.shape == (8, DIM) and np.isfinite(part).all(),
         'Oz and Fp1 zero-filled, still runs'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e3:.0f} KB)')
        print('Download it from Drive and upload to Codabench.')
    else:
        print('\nNot zipping: a check failed.')
